# Clase 7 · Notebook 01
# El bloque residual con la API funcional de Keras

**Introducción al Deep Learning — Módulo 3, Unidad 2: Redes de neuronas residuales**

Las redes residuales (**ResNet**) añaden **atajos** (skip connections) que suman la entrada a la salida de un
bloque. Esto evita el **desvanecimiento del gradiente** y permite entrenar redes muy profundas.

Como un bloque residual NO es una capa estándar, lo construiremos con la **API funcional** de Keras (no `Sequential`).

1. La API funcional: conectar capas con libertad.
2. Definir un bloque residual (Conv-BN-ReLU ×2 + suma).
3. Montar una pequeña CNN residual y entrenarla.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Preparar los datos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
tf.random.set_seed(42); np.random.seed(42)

digits = load_digits()
X = digits.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0
y = tf.keras.utils.to_categorical(digits.target, 10)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=digits.target)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. Definir un bloque residual

El bloque aplica dos convoluciones (con `BatchNormalization` y `ReLU`) y, al final, **suma la entrada original**
(el atajo) mediante la capa `Add`. La clave es `Add()([x, atajo])`.

In [ ]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Add, Input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def bloque_residual(x, filtros):
    atajo = x                                  # guardamos la entrada (skip connection)
    # Camino principal: Conv -> BN -> ReLU -> Conv -> BN
    y = Conv2D(filtros, (3, 3), padding="same")(x)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)
    y = Conv2D(filtros, (3, 3), padding="same")(y)
    y = BatchNormalization()(y)
    # Suma del atajo y activación final
    y = Add()([y, atajo])                      # F(x) + x
    y = Activation("relu")(y)
    return y

print("Bloque residual definido.")

## 3. Construir la red con la API funcional

En la API funcional, cada capa se llama sobre la anterior y guardamos el resultado en una variable.
Eso nos permite crear el atajo, algo imposible con `Sequential`.

In [ ]:
entrada = Input(shape=(8, 8, 1))
x = Conv2D(16, (3, 3), padding="same", activation="relu")(entrada)

x = bloque_residual(x, 16)     # primer bloque residual
x = bloque_residual(x, 16)     # segundo bloque residual

x = GlobalAveragePooling2D()(x)
salida = Dense(10, activation="softmax")(x)

modelo = Model(entrada, salida)
modelo.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

## 4. Entrenar

In [ ]:
historia = modelo.fit(X_train, y_train, epochs=15, batch_size=32,
                      validation_split=0.1, verbose=0)
acc = modelo.evaluate(X_test, y_test, verbose=0)[1]
print("Accuracy en test: %.1f %%" % (acc * 100))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(historia.history["accuracy"], label="entrenamiento", color="#000F9F")
plt.plot(historia.history["val_accuracy"], label="validación", color="#FF647E")
plt.title("Exactitud de la CNN residual"); plt.xlabel("Época"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

La red con bloques residuales entrena de forma estable. En redes mucho más profundas, los atajos son
los que hacen posible que el entrenamiento no se estanque.

## Resumen

- Un **bloque residual** calcula `F(x)` y le **suma la entrada**: `salida = F(x) + x` (capa `Add`).
- El atajo da al gradiente un **camino directo**, evitando que se desvanezca.
- Se construye con la **API funcional** de Keras (`Model`), no con `Sequential`.
- Existen versiones v1 y v2 según dónde se coloque la normalización.

➡️ En el **Notebook 02** veremos el **transfer learning**: reutilizar una red ya entrenada.